# H6. Exploratory checks (not pre-registered)

These checks were added after the hardware runs, to read the pre-registered results of H5. None of them is an
endpoint. Each one is a step of the protocol described in the paper, or a sensitivity check.

| Section | What it asks |
|---|---|
| H6.1 | How much of the noiseless signal survives on the device, against the shot-noise ceiling? |
| H6.2 | Can a classifier tell which job, or which session, a row came from? (batch identifiability) |
| H6.3 | Does the outcome-sorted leak survive a job-purged backtest? |
| H6.4 | AUC with 95% intervals per input, paired differences, and a hybrid input (classical + hardware) |
| H6.5 | Deviation 2: sessions 2 and 3 shared one two-qubit calibration. Does it matter? |

Needs the hardware outputs in `outputs/hardware/`. Run time: about 1 hour.

In [ ]:
N_BOOT = 2000
N_PERM = 200

In [ ]:
import os, sys, json, time, math, warnings
from pathlib import Path
# Find the study folder (the one holding the hsbcrep package), wherever Jupyter or VS Code started
def _find_root():
    starts = [Path(p).parent for p in [globals().get("__vsc_ipynb_file__")] if p]
    starts += [Path.cwd()]
    for s in starts:
        for p in [s, *s.parents]:
            if (p / "hsbcrep" / "__init__.py").exists():
                return p.resolve()
    raise RuntimeError(f"Cannot find the repository folder (the one holding hsbcrep/) from {Path.cwd()}. Set ROOT by hand in this cell.")
ROOT = _find_root()
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("repository folder:", ROOT)
OUT = ROOT / "outputs"; OUT.mkdir(exist_ok=True)
RAW = ROOT / "data" / "raw"
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
pd.set_option("display.width", 200)
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold, permutation_test_score
from hsbcrep import backtest as BT, design as D
BT.N_JOBS = 1

plan = json.loads((OUT / "prereg" / "plan.json").read_text()); NB = plan["backtest"]; SD = 1 / np.sqrt(plan["shots"])
ev = pd.read_csv(OUT / "events.csv"); F = pd.read_csv(OUT / "features.csv"); hw = np.load(OUT / "hw_events.npy")
NL = np.load(OUT / "noiseless_longer129_hw.npy"); POS = {e: k for k, e in enumerate(hw)}
FAST = {k: BT.families()[k] for k in ("LR", "XGB")}
HWD = OUT / "hardware"; log = json.loads((HWD / "log.json").read_text()); recs = []
for k, s in log["sessions"].items():
    if k == "0":
        continue                                                       # dry run, not scored
    for j in s["jobs"]:
        X = np.load(HWD / f"s{k}_j{j['job']}_evs.npy"); S = np.load(HWD / f"s{k}_j{j['job']}_std.npy")
        for i, (e, a) in enumerate(zip(j["events"], j["arms"])):
            recs.append({"session": int(k), "job": j["job"], "arm": a, "event": e, "X": X[i].ravel(), "sd": S[i].ravel()})
R = pd.DataFrame(recs)
print(R.groupby("arm").size().to_dict())

def run(X, e, blind=0, g=None, purge=False):
    """HSBC rolling backtest (as in H5). With purge=True, training rows from the test block's own jobs are dropped."""
    o = np.argsort(ev.t.values[e], kind="stable"); X, e = X[o], e[o]; g = g[o] if g is not None else None
    n = len(e); nt, bl = int(round(NB["train_fraction"] * n)), max(8, int(round(NB["block_fraction"] * n)))
    y, t = ev.y.values[e], ev.t.values[e]; rows = []
    for b0 in range(0, n, bl):
        b1 = min(b0 + bl, n)
        avail = np.flatnonzero(t < t[b0] - int(blind * 86_400_000)) if blind > 0 else np.arange(b0)
        if purge:
            avail = avail[~np.isin(g[avail], np.unique(g[b0:b1]))]
        if len(avail) < nt or len(np.unique(y[avail[-nt:]])) < 2:
            continue
        tr = avail[-nt:]; m, _, _ = BT._fit_best(X[tr], y[tr], FAST, 0)
        rows += [{"event": e[i], "p": p, "y": y[i]} for p, i in zip(m.predict_proba(X[b0:b1])[:, 1], range(b0, b1))]
    return pd.DataFrame(rows)

def auc_ci(p):
    rng = np.random.default_rng(0); y, s = p.y.values, p.p.values; d = []
    for _ in range(N_BOOT):
        ix = rng.integers(0, len(p), len(p))
        if y[ix].min() != y[ix].max():
            d.append(roc_auc_score(y[ix], s[ix]))
    lo, hi = np.percentile(d, [2.5, 97.5]); return round(roc_auc_score(y, s), 3), round(lo, 3), round(hi, 3)

def diff_ci(p1, p2):
    m = p1.merge(p2, on="event", suffixes=("1", "2")); y = m.y1.values; rng = np.random.default_rng(0); d = []
    for _ in range(N_BOOT):
        ix = rng.integers(0, len(m), len(m))
        if y[ix].min() != y[ix].max():
            d.append(roc_auc_score(y[ix], m.p1.values[ix]) - roc_auc_score(y[ix], m.p2.values[ix]))
    lo, hi = np.percentile(d, [2.5, 97.5])
    return round(roc_auc_score(y, m.p1) - roc_auc_score(y, m.p2), 3), round(lo, 3), round(hi, 3)

## H6.1 Device against noiseless, and the shot-noise ceiling

Per feature, across the R arm's events: the slope of hardware on noiseless values (signal kept), their correlation,
and the correlation one would see if the device only added shot noise (from the reported standard errors).

In [ ]:
Dr = R[R.arm == "R"]; H = np.stack(Dr.X.values); Sd = np.stack(Dr.sd.values); E = NL[[POS[e] for e in Dr.event]].reshape(len(Dr), -1)
Ec, Hc = E - E.mean(0), H - H.mean(0); ok = Ec.std(0) > 1e-3
slope = (Hc[:, ok] * Ec[:, ok]).sum(0) / (Ec[:, ok] ** 2).sum(0)
r_obs = (Ec[:, ok] * Hc[:, ok]).sum(0) / np.sqrt((Ec[:, ok] ** 2).sum(0) * (Hc[:, ok] ** 2).sum(0))
shot = np.sqrt((Sd[:, ok] ** 2).mean(0))
r_ceiling = np.abs(slope) * Ec[:, ok].std(0) / np.sqrt(slope ** 2 * Ec[:, ok].var(0) + shot ** 2)
q2 = {"signal kept (slope), median": np.median(slope), "correlation with noiseless, median": np.median(r_obs),
      "correlation if only shot noise were added, median": np.median(r_ceiling),
      "residual sd / shot sd, median": np.median((Hc[:, ok] - slope * Ec[:, ok]).std(0) / shot)}
pd.Series(q2).round(3)

## H6.2 Batch identifiability

Can a cross-validated logistic regression tell which of a session's two jobs a row came from, or which of the eight
sessions? Null-arm rows share one input, so any success comes from the device alone. The classical features of the
same R rows are a negative control and must sit at chance.

In [ ]:
clf = make_pipeline(StandardScaler(), LogisticRegression(C=0.05, max_iter=5000)); cv = StratifiedKFold(5, shuffle=True, random_state=0)
def perm_test(X, y, scoring):
    s, perm, p = permutation_test_score(clf, X, y, cv=cv, scoring=scoring, n_permutations=N_PERM, random_state=0)
    return round(s, 3), round(perm.mean(), 3), round(p, 4)

rows = []
for arm, feat in (("N", "hardware"), ("R", "hardware"), ("R", "classical (control)")):
    for s in range(1, 9):
        Ds = R[(R.arm == arm) & (R.session == s)]
        X = np.stack(Ds.X.values) if feat == "hardware" else F.values[Ds.event.values]
        rows.append({"rows": f"{arm} arm, {feat}", "session": s, **dict(zip(("AUC", "chance", "p"), perm_test(X, Ds.job.values, "roc_auc")))})
jobid = pd.DataFrame(rows)
display(jobid.groupby("rows")[["AUC", "chance"]].median().round(3))
rows = []
for arm, feat in (("N", "hardware"), ("R", "hardware"), ("R", "classical (control)")):
    Da = R[R.arm == arm]; X = np.stack(Da.X.values) if feat == "hardware" else F.values[Da.event.values]
    rows.append({"rows": f"{arm} arm, {feat}", **dict(zip(("accuracy", "chance", "p"), perm_test(X, Da.session.values, "accuracy")))})
pd.DataFrame(rows)

Each job carries its own "barcode". The difference between a session's two jobs does not repeat from session to
session, and summed over all features it is many shot-noise widths.

In [ ]:
Nn = R[R.arm == "N"]
M = {(s, j): np.stack(Nn[(Nn.session == s) & (Nn.job == j)].X.values).mean(0) for s in range(1, 9) for j in (0, 1)}
dj = np.stack([M[(s, 1)] - M[(s, 0)] for s in range(1, 9)]); C = np.corrcoef(dj)
print(f"job1 - job0 differences, mean correlation across sessions: {C[~np.eye(8, dtype=bool)].mean():.3f}")
print("separation d' between the two jobs of each session (shot-noise units):",
      np.round([np.sqrt(np.sum((dj[s] / SD) ** 2)) for s in range(8)], 1))

## H6.3 Replay of the null slots, standard and job-purged backtests

The null slots of each session are given that market day's events in a chosen order (as in H5.6). The job-purged
backtest drops, from each training window, every row that shares a job with the test block.

In [ ]:
Nr = R[R.arm == "N"].reset_index(drop=True).reset_index().rename(columns={"index": "row"}); Nr["session"] -= 1
rows = []
for order in ("random", "chronological", "by symbol", "by outcome"):
    pairs = D.reassign(Nr, ev, hw, order, np.random.default_rng(0))
    rr = np.array([a for a, _ in pairs]); e = np.array([b for _, b in pairs])
    X = np.stack(Nr.X.values[rr]); g = Nr.session.values[rr] * 10 + Nr.job.values[rr]
    for purge in (False, True):
        a, lo, hi = auc_ci(run(X, e, g=g, purge=purge))
        rows.append({"submission order": order, "backtest": "job-purged" if purge else "standard", "AUC": a, "95% CI": f"{lo:.3f} to {hi:.3f}"})
replay = pd.DataFrame(rows); replay.to_csv(OUT / "results_purged.csv", index=False); replay

## H6.4 AUC per input, paired differences, and a hybrid input

In [ ]:
eR = R[R.arm == "R"].event.values
P = {"classical": run(F.values[eR], eR), "noiseless": run(NL[[POS[e] for e in eR]].reshape(len(eR), -1), eR)}
for a in ("R", "C", "N"):
    Da = R[R.arm == a]; P[a] = run(np.stack(Da.X.values), Da.event.values)
Rr = R[R.arm == "R"]; P["classical + R"] = run(np.hstack([F.values[Rr.event.values], np.stack(Rr.X.values)]), Rr.event.values)
display(pd.DataFrame({k: auc_ci(v) for k, v in P.items()}, index=["AUC", "lo", "hi"]).T)
pd.DataFrame({f"{a} - {b}": diff_ci(P[a], P[b]) for a, b in
              [("noiseless", "classical"), ("R", "noiseless"), ("R", "classical"), ("C", "R"), ("classical + R", "classical")]},
             index=["difference", "lo", "hi"]).T

## H6.5 Deviation 2: sessions 2 and 3 on one two-qubit calibration

If sharing a calibration made sessions 2 and 3 alike, their null-arm means would be closer than other consecutive
sessions'.

In [ ]:
Xn = np.stack(Nn.X.values); ms = {s: Xn[Nn.session.values == s].mean(0) for s in range(1, 9)}
print(f"sessions 2 vs 3: mean |difference| {np.abs(ms[2] - ms[3]).mean():.4f}")
print(f"other consecutive sessions: {np.mean([np.abs(ms[a] - ms[a + 1]).mean() for a in (1, 3, 4, 5, 6, 7)]):.4f}")